# TM1py: Creating and Modifying Objects

This module is the sixth chunk of the tm1py course. Readers are
assumed to have absorbed the earlier chunks: the mental model of
Objects vs Services from chunk 1, the connection facade from chunk 2,
the structural walk through the metadata from chunk 3, and the read
and write paths for cell data from chunks 4 and 5.

The audience remains TM1 expert. Cubes, dimensions, hierarchies,
elements, edges, and consolidations are taken for granted as TM1
concepts; the focus is on how to construct their tm1py representations
and persist them to the server.

This is the first chunk that constructs Objects. The order is
deliberate. Most tm1py work reads from cubes and writes cells; far
less of it builds new dimensions, cubes, or processes from scratch.
Learners who reach for object construction first tend to fight the
library, modelling TM1 from Python rather than reflecting what is
already on the server. Coming to construction sixth, after the read
and write patterns are routine, makes the construction patterns
follow naturally.

The chunk also relies on the by-name vs by-embedding distinction
from chunk 3. That asymmetry was a property of how the metadata is
returned to the reader. It now becomes a property of how the writer
must compose the same Objects. A `Cube` constructor takes dimension
names as strings; a `Dimension` constructor takes `Hierarchy`
instances. Without the chunk 3 model in mind, that asymmetry looks
arbitrary. With it, every constructor signature is predictable.

The closing topics introduce `update_or_create` and the broader
discipline of writing idempotent scripts: the practical answer to
"how do I run this twice without breaking things on the second run."
Idempotency matters more for object construction than for cell
writes, because object construction often runs as part of a
deployment script that should be safe to repeat.

The Sales Plan model from earlier chunks continues as the running
example. The new piece this chunk introduces is a `Scenario`
dimension, with elements `Best Case`, `Base`, and `Worst Case`
rolling up into `All Scenarios`, and a small `What If Plan` cube
that uses it.

---

## Topic list

1. The construction scenario
2. Element objects
3. Hierarchies as embedded structures
4. Dimensions built from hierarchies
5. Cubes built from dimension names
6. By-name vs by-embedding, revisited
7. Persisting: create, update, update_or_create
8. Idempotent deployment scripts
9. A complete example
10. Real-world design principles
11. Common mistakes

---

## 1. The construction scenario

Most tm1py work reads from a model that someone else built. The
cubes, dimensions, and hierarchies came into being through PAW,
through Architect, or through a TI process that the team's modeller
wrote. Python's role in those workflows is to read, transform, and
write cells, not to define the structure. Construction is rarer.

The situations where Python does construct structure are well
defined. Bulk loading a dimension whose elements come from an
external source (a chart of accounts from a SQL system, a region
list from an HR file). Setting up a small auxiliary cube that an
analytical pipeline writes intermediate results into. Maintaining a
"build script" that creates a model from scratch in a development
environment for testing or training. Migrating part of a model from
one server to another. Each of these is the right place to construct
Objects in Python.

The discipline is to construct only what cannot easily be created
through PAW. PAW remains the right tool for interactive modelling,
for one-off cube creation, and for any work where the modeller wants
to see the structure as it grows. Python is the right tool when the
construction is parameterized, repeatable, version-controlled, or
driven by data the modeller cannot easily type by hand.

The running example for this chunk is a `Scenario` dimension with
three leaf elements (`Best Case`, `Base`, `Worst Case`) rolling up
into a consolidated parent (`All Scenarios`), and a small `What If
Plan` cube that uses it alongside the existing Sales Plan
dimensions. The dimension is small enough to fit in a few lines of
construction code; the cube is small enough that the construction
shape is visible without distraction.

## 2. Element objects

The smallest piece of TM1 structure that has a Python representation
is the element. The class is `Element`, imported from
`TM1py.Objects`.

In [ ]:
from TM1py.Objects import Element

best_case  = Element(name="Best Case",  element_type="Numeric")
base       = Element(name="Base",       element_type="Numeric")
worst_case = Element(name="Worst Case", element_type="Numeric")
all_scen   = Element(name="All Scenarios", element_type="Consolidated")

The element type is one of the three TM1 element kinds: `Numeric`,
`String`, or `Consolidated`. The type can also be supplied through
`Element.Types.NUMERIC`, `Element.Types.STRING`, and
`Element.Types.CONSOLIDATED` for code that prefers the enum form.

An `Element` on its own is just data. Constructing it does not put
anything on the server. The element exists in Python until it is
embedded into a hierarchy and the hierarchy is sent to the server.
That decoupling, Object first, Service later, is the same shape that
applied to reads in chunk 3 and writes in chunk 5.

For most construction work, the standalone `Element` class is rarely
needed. The `Hierarchy.add_element` method (next topic) accepts a
name and a type directly and constructs the element internally. The
explicit class is useful when an element has to be passed around
before being placed, or when many elements are constructed from a
data source and added to several hierarchies.

## 3. Hierarchies as embedded structures

A hierarchy holds elements and the parent–child edges that connect
them. The class is `Hierarchy`, also from `TM1py.Objects`. The
constructor takes the hierarchy's name and the name of the
containing dimension.

In [ ]:
from TM1py.Objects import Hierarchy

scenario_hier = Hierarchy(name="Scenario", dimension_name="Scenario")

scenario_hier.add_element("Best Case",     "Numeric")
scenario_hier.add_element("Base",          "Numeric")
scenario_hier.add_element("Worst Case",    "Numeric")
scenario_hier.add_element("All Scenarios", "Consolidated")

scenario_hier.add_edge(parent="All Scenarios", component="Best Case",  weight=1)
scenario_hier.add_edge(parent="All Scenarios", component="Base",       weight=1)
scenario_hier.add_edge(parent="All Scenarios", component="Worst Case", weight=1)

`add_element` adds an element to the hierarchy's internal `elements`
dict, keyed by name. `add_edge` records a parent-child relationship
with a weight (typically 1 for sums, sometimes -1 for subtractions or
fractional values for weighted aggregations). The order of calls
does not matter; the server reads the final state, not the sequence
of operations that built it.

A hierarchy supports element attributes too. The class is
`ElementAttribute`, and attributes are added to the hierarchy via
`add_element_attribute`.

In [ ]:
from TM1py.Objects import ElementAttribute

scenario_hier.add_element_attribute(
    ElementAttribute(name="Description", attribute_type="String")
)

Attribute values are stored in the corresponding control cube
(`}ElementAttributes_Scenario` for this hierarchy) and are written
through the cell write paths from chunk 5, not through the hierarchy
itself.

A hierarchy on its own, like an element, is just data. The hierarchy
becomes real on the server only when the dimension that contains it
is created or updated, or when the hierarchy is sent through
`tm1.hierarchies.update_or_create`. The next topic shows how a
dimension wraps the hierarchy.

## 4. Dimensions built from hierarchies

A `Dimension` is constructed with a name and a list of hierarchies.
The list takes `Hierarchy` instances directly, by embedding, because
a hierarchy is wholly owned by its containing dimension. This is the
first half of the asymmetry that topic 6 will name explicitly.

In [ ]:
from TM1py.Objects import Dimension

scenario_dim = Dimension(
    name="Scenario",
    hierarchies=[scenario_hier],
)

For a dimension with multiple hierarchies (a default and one or more
alternates), the list contains one `Hierarchy` per. For the common
case of a single hierarchy named the same as the dimension, the list
contains one entry. The library does not require the default
hierarchy to be present, but most TM1 tooling does, so creating a
dimension without a default hierarchy of the same name is rarely
the right choice.

A dimension is just data until it is sent. To create it on the
server:

In [ ]:
with TM1Service(**creds) as tm1:
    tm1.dimensions.create(scenario_dim)

`create` raises if a dimension with the same name already exists.
For idempotent scripts, `update_or_create` is preferable; topic 7
covers the verbs and topic 8 covers the discipline.

## 5. Cubes built from dimension names

A `Cube` is constructed with a name and a list of dimension names,
not a list of `Dimension` objects. This is the second half of the
asymmetry: a cube refers to its dimensions by name because dimensions
are independent and shared across cubes, and embedding them
redundantly would force the same dimension to appear in multiple
copies inside multiple cubes.

In [ ]:
from TM1py.Objects import Cube

what_if_plan = Cube(
    name="What If Plan",
    dimensions=[
        "Period",
        "Region",
        "Product",
        "Scenario",
        "Measure",
    ],
)

The dimensions named must already exist on the server when the cube
is created. A cube cannot bring its dimensions into being; it can
only assemble them. The construction order for a build script that
includes both new dimensions and new cubes is therefore: dimensions
first, cubes last.

A cube can also carry rules text:

In [ ]:
what_if_plan = Cube(
    name="What If Plan",
    dimensions=["Period", "Region", "Product", "Scenario", "Measure"],
    rules=(
        "['Best Case']  = ['Base'] * 1.20;\n"
        "['Worst Case'] = ['Base'] * 0.85;\n"
    ),
)

Rules text is a string, the same content that PAW or Architect would
show in the rules editor. Rules can also be set or replaced after
the cube exists by reading the cube, mutating its `rules` attribute,
and calling `tm1.cubes.update`. For non trivial rule logic, keeping
the rules in a separate `.rul` file under version control and
loading it into the `rules=` parameter at construction time is the
practical pattern.

## 6. By-name vs by-embedding, revisited

The previous two topics surfaced the asymmetry that chunk 3 named
when describing reads. It is worth restating in the construction
context, because the same rule that determined what `cube.dimensions`
returned now determines what the `Cube(...)` constructor accepts.

`Cube(dimensions=[...])` accepts a list of strings, the dimension
names. This is by-name reference. The reason: a dimension is
independent of any particular cube and can appear in many cubes at
once. The cube refers to the dimension by name; the dimension lives
in its own service.

`Dimension(hierarchies=[...])` accepts a list of `Hierarchy`
instances. This is by-embedding. The reason: a hierarchy is wholly
owned by its containing dimension. Embedding it directly costs
nothing in deduplication and saves the user from constructing the
hierarchy in one place and registering it with the dimension by
name in another.

The same pattern applies to other Objects covered in later chunks.
A `Process` constructor accepts its parameters as embedded objects;
parameters belong to the process. A `View` constructor accepts its
row, column, and title elements as embedded set definitions; the
slice belongs to the view. A `Subset` constructor accepts an element
list as embedded data; the subset belongs to its hierarchy.

The unifying rule is the one from chunk 3: tm1py refers by name when
the related thing is independent, and by embedding when the related
thing is wholly owned. Once the rule is internalized, the question
"does this constructor want strings or objects here?" answers itself
from the underlying TM1 ownership model, with no need to look at the
signature. The few apparent exceptions, fields that take a name when
embedding might have been more convenient, exist for performance
reasons (avoiding redundant payloads) and follow the same rule.

## 7. Persisting: create, update, update_or_create

A constructed Object lives in Python until it is sent to the server.
The verbs that send it are on the matching Service: `create`,
`update`, and `update_or_create`. Each has a clear role.

`create` sends the Object as a new entity. If an entity with that
name already exists, the call raises an error. `create` is
appropriate for scripts that genuinely should fail when re-run on
top of existing state, which is rare. The more common case is a
script that should be safe to run multiple times against a server
that may or may not already have the entity, in which case
`update_or_create` is the right call.

In [ ]:
with TM1Service(**creds) as tm1:
    tm1.dimensions.create(scenario_dim)        # raises if Scenario exists

`update` sends the Object as a modification to an existing entity.
If no entity with that name exists, the call raises an error. The
typical workflow is `tm1.dimensions.get(name)` to fetch the current
state, mutate the returned Object in Python, then `update` to send
the modified version back. The snapshot caveat from chunk 1 applies
here in full: between the `get` and the `update`, another writer's
change can be lost.

In [ ]:
with TM1Service(**creds) as tm1:
    dim = tm1.dimensions.get("Scenario")
    dim.hierarchies[0].add_element("Stretch", "Numeric")
    tm1.dimensions.update(dim)

`update_or_create` is the verb that makes deployment scripts
idempotent. If the entity exists, it is updated; if it does not, it
is created. Either way, the post-condition is "the server holds the
state described by the Object." Most build scripts should reach for
this verb almost exclusively.

In [ ]:
with TM1Service(**creds) as tm1:
    tm1.dimensions.update_or_create(scenario_dim)
    tm1.cubes.update_or_create(what_if_plan)

`update_or_create` is exposed on most services that have an
identifiable named entity: dimensions, cubes, hierarchies,
processes, chores, views, subsets. The pattern across the library is
predictable, and the verb name is searchable; when in doubt, the
relevant service has it.

## 8. Idempotent deployment scripts

A build script that creates a `Scenario` dimension and a `What If
Plan` cube is not a one-time operation. It runs in development. It
runs in CI, against an ephemeral test server. It runs again when a
developer re-deploys after a change. It runs in production once,
then again next quarter when the model is rebuilt elsewhere. Each
run must not break the server's state, and each run must leave the
server in the same final state regardless of starting state.

The discipline that achieves this is idempotency: run the script
twice, get the same result as running it once. For object
construction, the practical rules are:

Use `update_or_create` rather than `create`. The first time, it
creates; the second time, it updates with the same definition,
which is a no-op. Either way, the post-condition holds.

Never assume the object does not already exist. A script that calls
`create` and crashes if the dimension is there is not idempotent;
the next run leaves the server in an unrecoverable state because the
script cannot get past the first failing call.

For destructive changes (removing an element, dropping a hierarchy,
deleting a cube), separate the destruction from the construction.
Build scripts that recreate state from scratch should consider
whether the destruction is safe and reversible. A `delete` followed
by an `update_or_create` is technically idempotent, but every run
flushes any data that lived under that name. For production, this is
usually too aggressive; for ephemeral test servers, it is fine.

For state that depends on order (cubes that need their dimensions to
exist first), structure the script so that dimensions are persisted
before cubes that reference them. The construction order in the
running example is: build the `Scenario` dimension, then the `What
If Plan` cube. Reversing that order produces a server-side error on
cube creation because the dimension referenced by name does not
exist yet.

For configuration that may differ between environments (dev,
staging, prod), parameterize the script. The Object construction
takes parameters; the deployment is a function from environment to
the right `TM1Service` and the right parameter values. The same
script then deploys the same model, with environment-appropriate
specifics, against any of the targets.

## 9. A complete example

Putting the topics together, the deployment of the `Scenario`
dimension and the `What If Plan` cube fits in one self-contained
script.

In [ ]:
from TM1py import TM1Service
from TM1py.Objects import (
    Cube, Dimension, ElementAttribute, Hierarchy,
)


def build_scenario_dimension() -> Dimension:
    hier = Hierarchy(name="Scenario", dimension_name="Scenario")
    for leaf in ("Best Case", "Base", "Worst Case"):
        hier.add_element(leaf, "Numeric")
    hier.add_element("All Scenarios", "Consolidated")
    for leaf in ("Best Case", "Base", "Worst Case"):
        hier.add_edge(parent="All Scenarios", component=leaf, weight=1)
    hier.add_element_attribute(
        ElementAttribute(name="Description", attribute_type="String")
    )
    return Dimension(name="Scenario", hierarchies=[hier])


def build_what_if_cube() -> Cube:
    return Cube(
        name="What If Plan",
        dimensions=["Period", "Region", "Product", "Scenario", "Measure"],
        rules=(
            "['Best Case']  = ['Base'] * 1.20;\n"
            "['Worst Case'] = ['Base'] * 0.85;\n"
        ),
    )


def deploy(creds: dict) -> None:
    with TM1Service(**creds) as tm1:
        tm1.dimensions.update_or_create(build_scenario_dimension())
        tm1.cubes.update_or_create(build_what_if_cube())


if __name__ == "__main__":
    import os
    deploy({
        "address":  os.environ["TM1_ADDRESS"],
        "port":     int(os.environ["TM1_PORT"]),
        "user":     os.environ["TM1_USER"],
        "password": os.environ["TM1_PASSWORD"],
        "ssl":      True,
    })

The script can be run any number of times against any number of
servers and leaves them all in the same state: a `Scenario`
dimension with the right elements and edges, a `What If Plan` cube
that uses it, and a rule that derives the `Best Case` and `Worst
Case` scenarios from the `Base` scenario. The first run creates;
subsequent runs update with the same definition, which is a no-op.

The two construction functions are pure: they take no arguments,
read no state from the server, and return Object instances. The
`deploy` function is the boundary between pure Object construction
and the server side effect. Separating the two keeps the
construction unit-testable in isolation, with no TM1 connection
required, and the deployment small enough that its correctness can
be inspected by reading.

## 10. Real-world design principles

**Reflect, do not invent.** When a dimension or cube already exists
on the server, the right path is to read it (`tm1.dimensions.get`),
mutate, and update. Building it from scratch in Python and pushing
it back risks losing whatever was there before that the script
forgot to model. Construction is for things that need to exist
fresh, not for things that already do.

**Use update_or_create as the default.** It is idempotent, it is
the same call regardless of whether the entity already exists, and
it makes deployment scripts safe to re-run. Reach for `create` only
when the script genuinely should fail if the entity exists, and for
`update` only when the entity is known to exist and the change is
incremental.

**Construct dimensions before cubes that reference them.** A cube's
dimensions must exist on the server when the cube is created. The
construction order in any build script is therefore: elements
inside hierarchies, hierarchies inside dimensions, dimensions before
cubes. The reverse order produces an error.

**Separate construction from deployment.** Object construction is
pure: it takes parameters, it returns Objects, it touches no
external state. Deployment is impure: it talks to a server. Keeping
the two in different functions makes the construction
unit-testable, makes the deployment small enough to read at a
glance, and makes the script reusable across environments.

**Keep rules text in version-controlled files.** A non trivial
rules string in source code is hard to read, hard to diff, and hard
to maintain. The disciplined pattern is to keep `Sales_Plan.rul`
under version control and load it from disk into the `rules=`
parameter when the cube is constructed.

**Prefer building structure in PAW for one-off work.** Python
construction is the right tool when the structure is parameterized,
repeated, or driven by external data. For exploratory modelling,
PAW remains faster and more visual. The discipline is to know
which situation is which and to use the matching tool.

## 11. Common mistakes

A short collection of errors that come up while learning to
construct TM1 objects through tm1py.

**Passing `Dimension` objects to `Cube(dimensions=[...])`.** The
`Cube` constructor expects strings, not objects. Passing
`Dimension` instances either silently produces a malformed payload
or fails with a confusing error.

In [ ]:
# Wrong: passes Dimension objects
cube = Cube(
    name="What If Plan",
    dimensions=[period_dim, region_dim, product_dim, scenario_dim, measure_dim],
)

# Correct: pass dimension names
cube = Cube(
    name="What If Plan",
    dimensions=["Period", "Region", "Product", "Scenario", "Measure"],
)

**Passing strings to `Dimension(hierarchies=[...])`.** The
`Dimension` constructor expects `Hierarchy` instances, not names.
Passing names produces an error or a malformed dimension.

In [ ]:
# Wrong: passes hierarchy names
dim = Dimension(name="Scenario", hierarchies=["Scenario"])

# Correct: pass Hierarchy instances
hier = Hierarchy(name="Scenario", dimension_name="Scenario")
hier.add_element("Best Case", "Numeric")
# ... add elements and edges ...
dim = Dimension(name="Scenario", hierarchies=[hier])

**Calling `create` in a script that may run twice.** The second
run fails. The deployment script is not idempotent. The remedy is
`update_or_create`.

In [ ]:
# Wrong: fails on the second run if the dimension exists
tm1.dimensions.create(scenario_dim)

# Correct: idempotent
tm1.dimensions.update_or_create(scenario_dim)

**Building cubes before their dimensions.** A cube cannot reference a
dimension that does not exist. Construction order matters; the
script that creates the cube first and the dimensions afterwards
fails on the cube creation.

In [ ]:
# Wrong: cube references a dimension that does not exist yet
tm1.cubes.update_or_create(what_if_plan)
tm1.dimensions.update_or_create(scenario_dim)

# Correct: dimensions first, then cubes
tm1.dimensions.update_or_create(scenario_dim)
tm1.cubes.update_or_create(what_if_plan)

**Mutating a fetched Object without updating it.** Reading a
dimension, mutating it in Python, and forgetting the `update` call
leaves the server unchanged. The Python object reflects the
intended new state; the server still holds the old state.

In [ ]:
# Wrong: mutation never reaches the server
dim = tm1.dimensions.get("Scenario")
dim.hierarchies[0].add_element("Stretch", "Numeric")
# end of script, no update call

# Correct: send the mutated Object back
dim = tm1.dimensions.get("Scenario")
dim.hierarchies[0].add_element("Stretch", "Numeric")
tm1.dimensions.update(dim)

**Read-modify-write across a long gap on a shared cube.** The
chunk 1 snapshot rule applies in full. A long gap between `get` and
`update` lets another writer's change get overwritten silently.

In [ ]:
# Wrong: long gap, shared cube, no conflict detection
dim = tm1.dimensions.get("Scenario")
# ... five minutes of computation, user prompts, etc. ...
dim.hierarchies[0].add_element("Stretch", "Numeric")
tm1.dimensions.update(dim)
# any change made in the gap is overwritten

# Correct: minimize the gap, or model the change as a small delta
dim = tm1.dimensions.get("Scenario")
dim.hierarchies[0].add_element("Stretch", "Numeric")
tm1.dimensions.update(dim)        # send immediately

**Embedding rules text directly in source for non trivial rules.**
A multiline rules string in a Python literal is hard to read and
maintain. Loading from a file is one extra line and is cleaner.

In [ ]:
# Wrong: large rules block embedded in code
cube = Cube(
    name="What If Plan",
    dimensions=[...],
    rules="""
    [...] = ...;
    [...] = ...;
    ...
    """,
)

# Correct: rules in their own file, version-controlled separately
with open("rules/what_if_plan.rul", "r") as f:
    rules_text = f.read()

cube = Cube(
    name="What If Plan",
    dimensions=[...],
    rules=rules_text,
)

**Forgetting that element and dimension names are case sensitive.**
A `Cube(dimensions=["scenario"])` referencing a dimension created
as `Scenario` produces an error. The rule from chunk 3 applies
without exception.

In [ ]:
# Wrong: lowercase reference does not match the dimension's name
cube = Cube(
    name="What If Plan",
    dimensions=["Period", "Region", "Product", "scenario", "Measure"],
)
# fails: dimension 'scenario' not found

# Correct: canonical casing
cube = Cube(
    name="What If Plan",
    dimensions=["Period", "Region", "Product", "Scenario", "Measure"],
)